# Open-XTTS-Trainer: End-to-End 3-Stage Training Demo

This notebook demonstrates how to train or fine-tune all 3 stages of **XTTS v2**:
1. **Stage 2:** Text Tokenizer & Vocabulary Extension for New Scripts (e.g. Arabic, Hindi, Cyrillic)
2. **Stage 3:** Latent HiFi-GAN Vocoder (Extracting GPT latents & training generator + discriminator)
3. **Stage 1:** Discrete VAE (DVAE) Audio Tokenizer fine-tuning

## 1. Setup & Environment

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

# Check GPU availability
!nvidia-smi

# Install open-xtts-trainer and Idiap's Coqui TTS fork
!pip install -q git+https://github.com/idiap/coqui-ai-TTS.git
!git clone https://github.com/amitbhakt/open-xtts-trainer.git 2>/dev/null || (cd open-xtts-trainer && git pull)
%cd open-xtts-trainer
!pip install -q -e .

## 2. Stage 2: GPT Alignment & Vocabulary Extension


In [ ]:
# 1. Download official XTTS v2 vocab.json
!mkdir -p xtts_base
!wget -q -O xtts_base/vocab.json https://huggingface.co/coqui/XTTS-v2/raw/main/vocab.json

# 2. Sample target language text (e.g. Arabic, Hindi, Dhivehi)
with open("sample_target.txt", "w", encoding="utf-8") as f:
    f.write("ދިވެހި ބަސް. สวัสดี. مرحبا.")

# 3. Extend Tokenizer for new alphabet
!python xtts_trainer/stage2_gpt/vocab_extender.py --base_vocab xtts_base/vocab.json --target_text sample_target.txt --output_vocab xtts_base/vocab_extended.json

# 4. Fine-Tune Autoregressive GPT Backbone (Text-to-Speech Alignment)
!python xtts_trainer/stage2_gpt/train.py --audio_dir sample_audio --vocab_path xtts_base/vocab_extended.json --output_dir checkpoints/gpt --epochs 5 --device cuda


## 3. Stage 3: Latent HiFi-GAN Vocoder Training on Real Speech

In [ ]:
# 1. Download & Prepare 5.0 Minutes of Real Human Speech (100 training clips)
!rm -rf sample_audio extracted_latents
!python -m xtts_trainer.speech_data --minutes 5 --output_dir sample_audio

# 2. Extract GPT Conditioning Latents & Speaker Embeddings (~25s on GPU)
!python xtts_trainer/stage3_hifigan/extract_latents.py --audio_dir sample_audio --output_dir extracted_latents --device cuda

# 3. Train Latent HiFi-GAN Vocoder (25 Epochs ~ 300 steps on 100 human speech clips)
!python xtts_trainer/stage3_hifigan/train.py --audio_dir sample_audio --latents_dir extracted_latents --output_dir checkpoints/hifigan --batch_size 8 --epochs 25 --device cuda


## 4. Stage 1: Discrete VAE (DVAE) Fine-Tuning

In [ ]:
# Download official dvae.pth checkpoint to fine-tune from
!wget -q -O xtts_base/dvae.pth https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth

# Run DVAE training/fine-tuning on the 100 speech clips
!python xtts_trainer/stage1_dvae/train.py --audio_dir sample_audio --output_dir checkpoints/dvae --continue_path xtts_base/dvae.pth --batch_size 8 --epochs 25 --device cuda


## 5. View Checkpoints & Play Audio Results

In [ ]:
# 1. Find and list all saved model checkpoints across all 3 stages
print("=== Locate Checkpoints on Disk ===")
!find /content -name "*.pt" -o -name "*.pth" 2>/dev/null

# 2. View Benchmark Loss Convergence Chart on Real Human Speech
from IPython.display import Image, display
display(Image("assets/benchmark_human_speech.png"))

# 3. Play sample speech directly in Colab
import IPython.display as ipd
sample_wav = !find /content -name "speech_001.wav" -o -name "speech_*.wav" 2>/dev/null | grep -v "checkpoint" | head -n 1
if sample_wav:
    print("Playing:", sample_wav[0])
    display(ipd.Audio(sample_wav[0]))
